# 17. Sklearn Residual Stack

15번의 leakage-safe 예측을 기준으로 두고, LightGBM/CatBoost와 다른 계열의 sklearn tree model이 남은 residual을 설명할 수 있는지 검증합니다.

- Test는 최종 `transform`/`predict`와 제출 ID 정합성 확인에만 사용합니다.
- Residual stacker는 시간 순서를 지켜 Fold2는 Fold1 residual만, Fold3는 Fold1+2 residual만 사용합니다.
- OOF에서 안정적으로 개선될 때만 제출 파일을 생성합니다.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 140)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

# 15번 노트북을 기준 예측 생성기로 재실행합니다.
# 15 내부는 fold-fit/past-only 검증과 Train-only final fit을 사용합니다.
nb15 = json.loads((PROJECT_ROOT / 'notebooks' / '15_market_correction_refined.ipynb').read_text())
code15 = ''.join(nb15['cells'][1]['source'])
ns = {}
exec(code15, ns)

rmse = ns['rmse']
rmsle = ns['rmsle']
train = ns['train']
test = ns['test']
sample_submission = ns['sample_submission']
SUBMISSION_DIR = ns['SUBMISSION_DIR']
TIME_FOLDS = ns['TIME_FOLDS']

# 15 OOF frame: 각 fold의 validation row에 대해 OOF 기반 예측만 들어 있습니다.
oof15 = pd.concat(ns['history_market'], ignore_index=True).copy()
oof15['Residual15'] = oof15['Target'] - oof15['Pred15']

# 최종 Test frame: 15까지의 final prediction을 feature로만 사용합니다.
test_frame = ns['test_residual_frame_11'].copy()
test_frame['Pred09'] = ns['test_pred_09']
test_frame['Pred11'] = ns['test_pred_11']
test_frame['Pred15'] = ns['test_pred']
test_frame['BasePred_Brand'] = test_frame['BasePred_Bin'].astype(str) + '_' + test_frame['Brand_Apartment'].astype(str)

# sklearn stacker용 feature. Target/residual/fold 정보는 제외합니다.
DROP_COLS = [
    'Target', 'Residual', 'Residual05', 'Residual06', 'Residual08', 'Residual09', 'Residual15',
    'Fold', 'Pred04', 'Pred05', 'Pred06', 'Pred08'
]
BASE_FEATURES = [c for c in oof15.columns if c not in DROP_COLS]
BASE_FEATURES = [c for c in BASE_FEATURES if c in test_frame.columns or c in ['Pred09', 'Pred11', 'Pred15']]

CAT_COLS = [
    'Gu', 'Dong', 'Gu_Dong', 'Gu_AreaBin', 'Gu_BasePredBin', 'Brand_AreaBin',
    'BasePred_AgeBin', 'BasePred_Brand', 'Area_Bin', 'Age_Bin', 'BasePred_Bin',
]
CAT_COLS = [c for c in CAT_COLS if c in BASE_FEATURES]
NUM_COLS = [c for c in BASE_FEATURES if c not in CAT_COLS]

def make_preprocessor():
    try:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse=False, dtype=np.float32)
    return ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), NUM_COLS),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value='__missing__')),
                ('onehot', encoder),
            ]), CAT_COLS),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )

MODEL_SPECS = {
    'ExtraTrees_leaf5': ExtraTreesRegressor(
        n_estimators=700, min_samples_leaf=5, max_features=0.75,
        random_state=2026, n_jobs=-1, bootstrap=False,
    ),
    'ExtraTrees_leaf8': ExtraTreesRegressor(
        n_estimators=700, min_samples_leaf=8, max_features=0.70,
        random_state=2027, n_jobs=-1, bootstrap=False,
    ),
    'RandomForest_leaf8': RandomForestRegressor(
        n_estimators=600, min_samples_leaf=8, max_features=0.70,
        random_state=2028, n_jobs=-1, bootstrap=True,
    ),
    'HistGB_leaf15': HistGradientBoostingRegressor(
        max_iter=350, learning_rate=0.03, max_leaf_nodes=15, l2_regularization=20.0,
        min_samples_leaf=20, random_state=2029,
    ),
}

def fit_predict_residual(model, fit_frame, predict_frame):
    pipe = Pipeline([
        ('prep', make_preprocessor()),
        ('model', model),
    ])
    X_fit = fit_frame[BASE_FEATURES].copy()
    y_fit = fit_frame['Residual15'].to_numpy()
    X_predict = predict_frame[BASE_FEATURES].copy()
    pipe.fit(X_fit, y_fit)
    pred = pipe.predict(X_predict)
    lo, hi = np.percentile(y_fit, [10, 90])
    pred = np.clip(pred, lo, hi)
    return pred, pipe

def evaluate_spec(spec_name, alpha):
    rows = []
    pred17_all = []
    y_all = []
    for fold in sorted(oof15['Fold'].unique()):
        valid_frame = oof15.loc[oof15['Fold'] == fold].copy()
        y = valid_frame['Target'].to_numpy()
        pred15 = valid_frame['Pred15'].to_numpy()
        if fold == 1 or alpha == 0:
            residual_pred = np.zeros(len(valid_frame))
        else:
            fit_frame = oof15.loc[oof15['Fold'] < fold].copy()
            residual_pred, _ = fit_predict_residual(MODEL_SPECS[spec_name], fit_frame, valid_frame)
        pred17 = np.clip(pred15 + alpha * residual_pred, 0, None)
        rows.append({
            'Model': spec_name,
            'Alpha': alpha,
            'Fold': int(fold),
            'Rows': len(valid_frame),
            '15_RMSE': rmse(y, pred15),
            '17_RMSE': rmse(y, pred17),
            '17_vs_15_Improvement': rmse(y, pred15) - rmse(y, pred17),
            'Mean_abs_move': float(np.mean(np.abs(pred17 - pred15))),
        })
        pred17_all.extend(pred17)
        y_all.extend(y)
    return pd.DataFrame(rows), np.asarray(y_all), np.asarray(pred17_all)

alpha_grid = [0.00, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30]
summary_rows = []
detail_tables = {}
for spec_name in MODEL_SPECS:
    for alpha in alpha_grid:
        details, y_all, pred_all = evaluate_spec(spec_name, alpha)
        detail_tables[(spec_name, alpha)] = details
        base_all = []
        for fold in sorted(oof15['Fold'].unique()):
            base_all.extend(oof15.loc[oof15['Fold'] == fold, 'Pred15'].to_numpy())
        pooled_15 = rmse(y_all, base_all)
        pooled_17 = rmse(y_all, pred_all)
        eligible = details.loc[details['Fold'] > 1]
        summary_rows.append({
            'Model': spec_name,
            'Alpha': alpha,
            '15_Pooled_RMSE': pooled_15,
            '17_Pooled_RMSE': pooled_17,
            'Pooled_Improvement': pooled_15 - pooled_17,
            'Improved_Eligible_Folds': int((eligible['17_RMSE'] < eligible['15_RMSE']).sum()),
            'Worst_Eligible_Improvement': eligible['17_vs_15_Improvement'].min(),
            'Fold3_Improvement': details.loc[details['Fold'] == 3, '17_vs_15_Improvement'].iloc[0],
            'Mean_abs_move': details['Mean_abs_move'].mean(),
        })

summary = pd.DataFrame(summary_rows).sort_values(
    ['Pooled_Improvement', 'Worst_Eligible_Improvement', 'Fold3_Improvement'],
    ascending=False,
).reset_index(drop=True)
display(summary.head(20))

# 너무 공격적인 OOF 최적화는 public LB에서 흔들렸으므로 안정성 조건을 둡니다.
stable = summary.query(
    'Alpha > 0 and Pooled_Improvement > 0.75 and Improved_Eligible_Folds == 2 and Worst_Eligible_Improvement > 0.10 and Fold3_Improvement > 0.10'
).copy()

if len(stable) == 0:
    print('안정성 조건을 통과한 sklearn residual stack이 없습니다. 제출 파일을 생성하지 않고 15를 유지합니다.')
    best = summary.iloc[0].copy()
    display(detail_tables[(best['Model'], best['Alpha'])])
    CREATE_SUBMISSION = False
else:
    best = stable.iloc[0].copy()
    CREATE_SUBMISSION = True
    print('선택된 17 설정:')
    display(best.to_frame('value'))
    display(detail_tables[(best['Model'], best['Alpha'])])

# Leakage audit
leakage_audit = pd.Series({
    '15 baseline predictions reproduced with fold-fit/past-only logic': True,
    'Stacker fold validation uses only earlier OOF folds as fit data': True,
    'OneHot/Imputer fit only on each stacker fit frame': True,
    'Residual target uses OOF Pred15, not in-sample prediction': True,
    'Model/alpha selected by Train OOF only': True,
    'Test used only for final transform/predict and submission row check': True,
})
display(leakage_audit.rename('Passed'))
assert leakage_audit.all()

if CREATE_SUBMISSION:
    best_model_name = str(best['Model'])
    best_alpha = float(best['Alpha'])
    final_residual_pred, final_pipe = fit_predict_residual(MODEL_SPECS[best_model_name], oof15, test_frame)
    test_pred_17 = np.clip(ns['test_pred'] + best_alpha * final_residual_pred, 0, None)
    prediction_frame = pd.DataFrame({'ID': test['ID'], 'Target': test_pred_17})
    submission = sample_submission[['ID']].merge(prediction_frame, on='ID', how='left', validate='one_to_one')
    assert submission['Target'].notna().all()
    assert len(submission) == len(sample_submission)
    assert sample_submission['ID'].equals(submission['ID'])
    output_path = SUBMISSION_DIR / '17_sklearn_residual_stack_submission.csv'
    submission.to_csv(output_path, index=False)
    print(f'Saved: {output_path}')
    display(submission.head())
else:
    output_path = None

result = {
    'create_submission': CREATE_SUBMISSION,
    'selected_model': None if len(stable) == 0 else str(best['Model']),
    'selected_alpha': None if len(stable) == 0 else float(best['Alpha']),
    'selected_oof_improvement': float(best['Pooled_Improvement']),
    'submission_path': None if output_path is None else str(output_path),
}
print(result)


Train: (1969, 11)
Train period: 202401 ~ 202512


,Fold,Train_rows,Valid_rows,04_RMSE,05_RMSE,06_RMSE,08_RMSE,09_RMSE,09_RMSLE,08_vs_06_Improvement,09_vs_08_Improvement,Local_Applied_Rows,Second_Local_Applied_Rows,Third_Local_Applied_Rows
0,Train-derived Fold 1,1201,278,"2,107.743326","2,107.743326","2,107.743326","2,107.743326","2,107.743326",0.055311,0.000000,0.000000,0,0,0
1,Train-derived Fold 2,1479,244,"2,634.763161","2,628.945780","2,621.229339","2,618.051538","2,614.984309",0.080279,3.177802,3.067229,141,103,108
2,Train-derived Fold 3,1723,246,"2,525.601387","2,522.504744","2,522.382035","2,520.144861","2,516.302312",0.071534,2.237174,3.842550,146,100,100


04 pooled RMSE: 2,420.08
05 pooled RMSE: 2,417.04
06 pooled RMSE: 2,414.33
08 pooled RMSE: 2,412.49
09 pooled RMSE: 2,410.15
Local-corrected improved folds: 2/2
Second-local improved folds: 2/2
Third-local improved folds: 2/2
08 vs 06 pooled improvement: 1.84
09 vs 08 pooled improvement: 2.34


Imputation statistics fitted on Train/fold-fit only                  True
OneHotEncoder fitted on Train/fold-fit only                          True
No independent dummy encoding on Test                                True
Fixed bins avoid validation/test distribution fitting                True
Target encoding excludes each training row target                    True
Residual model trained from time-OOF residuals only                  True
Local residual maps fitted from previous OOF residuals only          True
Second local correction selected with strict Train OOF thresholds    True
Third local correction selected with strict Train OOF thresholds     True
Validation uses strictly earlier transactions                        True
Weights, shrinkage and local corrections selected without Test       True
Test limited to transform and predict                                True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/09_residual08_cluster_correction_submission.csv


...

,Model,Alpha,Fold,Rows,15_RMSE,17_RMSE,17_vs_15_Improvement,Mean_abs_move
0,HistGB_leaf15,0.080000,1,278,"2,107.743326","2,107.743326",0.000000,0.000000
1,HistGB_leaf15,0.080000,2,244,"2,592.242694","2,588.127758",4.114936,61.369429
2,HistGB_leaf15,0.080000,3,246,"2,511.384324","2,511.258274",0.126050,51.051441


15 baseline predictions reproduced with fold-fit/past-only logic       True
Stacker fold validation uses only earlier OOF folds as fit data        True
OneHot/Imputer fit only on each stacker fit frame                      True
Residual target uses OOF Pred15, not in-sample prediction              True
Model/alpha selected by Train OOF only                                 True
Test used only for final transform/predict and submission row check    True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/17_sklearn_residual_stack_submission.csv


,ID,Target
0,TR_2427,"36,937.484718"
1,TR_0028,"48,170.602450"
2,TR_1461,"29,150.967192"
3,TR_1977,"47,532.510021"
4,TR_2351,"46,969.224666"


{'create_submission': True, 'selected_model': 'HistGB_leaf15', 'selected_alpha': 0.08, 'selected_oof_improvement': 1.4532255532062663, 'submission_path': '/Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/17_sklearn_residual_stack_submission.csv'}
